# v1.0 — Deployment reproduzível: decisões e evidências

**Autor:** Wanderson Ferreira  
**Branch:** `infra/reproducible-deployment`  
**Marco operacional:** 05/10/2026. Notebook organizado em 06/10/2026 UTC.

Este notebook documenta a etapa extensa de infraestrutura da v1.0: reprodução do sistema em um segundo KinD, gestão da release por Terraform/Helm, recuperação do estado persistente e acesso por clientes isolados. O deployment local foi exercitado; o blueprint AWS permanece pendente e a v1.0 ainda não está encerrada.

O [notebook principal](../renewable-energy-mlops.ipynb) mantém a síntese da evolução. Este arquivo preserva decisões, resultados e contratempos desta branch. O [runbook](../../docs/REPRODUCIBLE_DEPLOYMENT.md) é a referência para comandos, pré-requisitos e execução. Os recibos públicos estão em [docs/evidence](../../docs/evidence/).

As saídas operacionais foram executadas pelo autor em seu host e consolidadas em JSONs sanitizados. As células abaixo inspecionam esses recibos históricos com a biblioteca padrão Python; não acessam clusters, não provisionam recursos e não repetem os testes de integração. Executá-las não demonstra que o ambiente atual está disponível.

## 1. Problema e decisão de arquitetura

A v0.3 já servia o champion v17, com governança same-OOT e acompanhamento longitudinal iniciado. A imagem da API já havia sido reduzida e implantada por digest. A pergunta desta etapa passou a ser: **conseguimos reconstruir o ambiente e recuperar o mesmo modelo, com um procedimento verificável?**

Criar workloads vazios não recupera o sistema MLOps. Foi necessário tratar três partes: declaração da infraestrutura, imagens executáveis e estado persistente do Registry/datasets/artefatos.

| Componente | Responsabilidade nesta etapa |
|---|---|
| KinD separado | Cluster de ensaio, node fixo e kubeconfig próprio |
| Helm | Templates dos workloads, serviços e PVCs |
| Terraform + provider Helm | Planejar e aplicar a release do ensaio |
| Scripts de snapshot/restore | Capturar e restaurar banco e objetos S3 |
| Gates | Conferir identidade, artefatos, inferência e convergência |
| Poetry no host | Runtime do cliente Prefect/Streamlit conforme lockfile |

O operador inicia o procedimento; rollout e validação estão integrados ao fluxo Terraform/Helm do ensaio. O Actions valida configurações e publica imagens, mas não implanta automaticamente no cluster do host. A origem conserva seu rollout manual. Portanto, o resultado não é descrito como GitOps completo.

A escolha mantém o trabalho sem gastos cloud: infraestrutura local executável agora; arquitetura AWS declarada e validada em uma frente posterior. LocalStack não foi utilizado neste marco.

## 2. Isolamento antes de qualquer escrita

| Fronteira | Origem operacional | Ensaio |
|---|---|---|
| Contexto | `kind-energy-mlops` | `kind-energy-mlops-repro` |
| Namespace | `default` | `energy-mlops-repro` |
| Release | `energy-mlops` | `energy-mlops-repro` |
| Kubeconfig do ensaio | — | `.repro/kubeconfig` |
| API / MLflow / RustFS | 8000 / 5000 / 9000 | 18000 / 15000 / 19000 |
| Prefect / dashboard | 4200 / 8501 | 14200 / 18501 |

O contexto operacional permaneceu selecionado após a criação do segundo cluster. Os wrappers usam o destino explícito e verificam o node do ensaio. A API do destino começa desabilitada: o Registry vazio ainda não possui um champion carregável.

As imagens foram fixadas por digest, incluindo o node KinD. Isso registra os artefatos usados no ensaio; não implica equivalência entre arquiteturas de CPU, nem atualização automática dessas imagens. Os PVCs são locais, `standard`, RWO: PostgreSQL 2 GiB e RustFS 5 GiB.

Secrets, state/plan Terraform, kubeconfig, snapshots e logs brutos permanecem privados e fora do Git. O notebook publica apenas informações sanitizadas e não contém credenciais.

## 3. Primeiro apply e convergência

O bootstrap criou uma release Terraform `helm_release.mlops`. PostgreSQL, RustFS e MLflow ficaram Ready, e ambos os PVCs ficaram Bound. O primeiro apply registrou **1 recurso adicionado, 0 alterados, 0 destruídos**, em 52 segundos. A contagem corresponde à release gerenciada pelo Terraform; ela contém vários objetos Kubernetes.

O plano posterior retornou `No changes`. Isso demonstra convergência da configuração da release naquela execução. A saúde dos workloads e a identidade do modelo são verificações complementares, feitas pelos gates.

Dois testes Terraform com mocks passaram no bootstrap. Eles exercitam o contrato da configuração; a evidência do provisionamento real vem do apply e dos comandos Kubernetes executados no host.

In [1]:
import json
from pathlib import Path

def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'docs/evidence').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('Execute este notebook dentro do repositório.')

repo_root = find_repo_root()

def receipt(name):
    return json.loads((repo_root / 'docs/evidence' / name).read_text(encoding='utf-8'))

bootstrap = receipt('terraform_bootstrap_2026-10-05.json')
assert bootstrap['context'] == 'kind-energy-mlops-repro'
assert bootstrap['api_enabled'] is False
assert bootstrap['apply']['resource'] == 'helm_release.mlops'
assert bootstrap['post_apply_plan']['result'] == 'No changes'
print(json.dumps({'evidence': 'historical bootstrap', 'apply': bootstrap['apply'],
                  'post_apply_plan': bootstrap['post_apply_plan']['result']}, indent=2))

{
  "evidence": "historical bootstrap",
  "apply": {
    "added": 1,
    "changed": 0,
    "destroyed": 0,
    "elapsed_seconds": 52,
    "resource": "helm_release.mlops"
  },
  "post_apply_plan": "No changes"
}


## 4. Recuperar o estado: PostgreSQL e objetos S3

O inventário inicial confirmou que a origem tinha modelos registrados e dois buckets, enquanto o destino estava vazio. O snapshot `snapshot-20261005T201656Z.LOW70R` reuniu dump PostgreSQL, objetos S3, manifestos e hashes.

| Bucket | Objetos | Bytes restaurados |
|---|---:|---:|
| `energy-lake` | 19 | 20.949.616 |
| `mlflow-artifacts` | 232 | 1.808.792.024 |
| Total | 251 | 1.829.741.640 |

O procedimento pede uma janela sem gravações de treino, ingestão, promoção ou monitoring. Isso importa porque banco e S3 são capturados separadamente: não há snapshot atômico entre os dois stores, nem backup PITR.

O restore revalida o snapshot antes de escrever, rejeita destino já preenchido e verifica os hashes dos objetos restaurados. Apenas o MLflow do ensaio é suspenso para restaurar o banco. Depois, o gate consulta o Registry e confirma o acesso aos artefatos.

Sete testes do snapshot e sete do restore passaram no host. Entre as fronteiras verificadas estão corrupção, mudança do snapshot, destino ocupado, alvo incorreto e falha do banco. A restauração real confirmou 251 objetos e preservou o modelo v17 e sua Run. O recibo de restore registra API ainda desabilitada; a etapa seguinte fecha essa pendência cronológica.

In [2]:
restore = receipt('snapshot_restore_2026-10-05.json')
expected_run = '1d13a61244c54f06aa70f43a9993ea37'
assert restore['verified_objects'] == 251
assert restore['verified_bytes'] == 1829741640
assert restore['model']['version'] == 17
assert restore['model']['run_id'] == expected_run
print(json.dumps({'evidence': 'historical restore', 'model': restore['model'],
                  'objects': restore['verified_objects'],
                  'bytes': restore['verified_bytes']}, indent=2))

{
  "evidence": "historical restore",
  "model": {
    "name": "ensemble_lgb_xgb_rf_bahia",
    "version": 17,
    "run_id": "1d13a61244c54f06aa70f43a9993ea37",
    "source": "models:/m-b5741525f9ce41c98d166288955e9f6b",
    "download_uri": "s3://mlflow-artifacts/6/models/m-b5741525f9ce41c98d166288955e9f6b/artifacts"
  },
  "objects": 251,
  "bytes": 1829741640
}


## 5. Serving: identidade e comparação pareada

Depois do restore, a API foi habilitada por Terraform/Helm. O gate confirmou `ensemble_lgb_xgb_rf_bahia`, alias `champion`, v17, Run `1d13a61244c54f06aa70f43a9993ea37`.

Um mesmo lote sintético de três horas foi enviado à origem e ao destino. As diferenças máximas foram **0,0 em `predicted_fc` e `predicted_mw`**, dentro das tolerâncias registradas. O lote usa uma referência própria (`paired_synthetic_batch_v1`); não deve ser confundido com o smoke anterior da otimização Docker, que tinha `predicted_mw = 2092.931369766858`.

Essa comparação valida reprodução da inferência para as entradas exercitadas. Não constitui treino novo, promoção adicional, medição de generalização ou validação day-ahead. As métricas OOT da v17 continuam sendo as da promoção same-OOT: MAE 787,1836 MW, nMAE 6,6788% e R² 0,8614.

## 6. Falha controlada e recuperação declarativa

O ensaio aplicou deliberadamente um digest inexistente à API. O pull falhou com `ErrImagePull` e o apply expirou no timeout controlado. A recuperação reaplicou a configuração com o digest válido por Terraform/Helm.

O gate pós-recuperação confirmou o champion, as previsões pareadas, as identidades dos PVCs e um plano final `No changes`. O aprendizado é que uma falha de rollout pode ser detectada e tratada dentro do procedimento declarado, preservando o estado persistente.

O escopo foi uma falha de pull da imagem da API no cluster isolado. Não foi exercitado rollback do runtime MLflow, migração reversa do banco ou troca funcional entre duas versões da API. Também não foi medida disponibilidade contínua durante a falha.

## 7. Runtime MLflow empacotado

A imagem original do MLflow dependia de instalação de pacotes no startup. Foi criada uma imagem própria, baseada no digest observado, com versões explícitas dos SDKs/driver PostgreSQL. O build verificou imports e `pip check`; o smoke HTTP usou SQLite temporário e rede externa desabilitada.

O Actions publicou a imagem. A adoção por digest no ensaio removeu a instalação no startup e foi seguida por consultas reais ao Registry, leitura/hash de um artefato `MLmodel`, preservação dos PVCs e novo gate de serving. O plano final ficou sem mudanças, com chart 0.1.5.

| Runtime | Contrato |
|---|---|
| Aplicação / cliente | Python 3.14.4 e dependências do `poetry.lock` |
| Servidor MLflow | Python 3.11.15 e versões empacotadas no runtime dedicado |

Esses ambientes têm responsabilidades e SDKs próprios; as versões boto3/botocore do cliente e do servidor não são iguais. O lockfile continua sendo a fonte de verdade da aplicação. A imagem Docker local e a publicada também não são declaradas byte a byte idênticas: a evidência operacional usa o digest publicado.

O gate do runtime consultou um `MLmodel`; a verificação completa dos 251 objetos pertence ao restore anterior. As duas evidências têm escopos complementares.

In [3]:
adoption = receipt('mlflow_runtime_adoption_2026-10-05.json')
gate = adoption['serving_gate']
assert gate['model']['version'] == 17
assert gate['model']['run_id'] == expected_run
left, right = gate['source_predictions'], gate['destination_predictions']
assert len(left) == len(right) == 3
differences = {}
for field in ('predicted_fc', 'predicted_mw'):
    assert all(a['date'] == b['date'] for a, b in zip(left, right))
    differences[field] = max(abs(a[field] - b[field]) for a, b in zip(left, right))
    assert differences[field] <= gate['absolute_tolerances'][field]
assert adoption['post_apply_plan']['result'] == 'No changes'
assert adoption['post_apply_plan']['mlflow_install_runtime_packages'] is False
recovery = receipt('deployment_recovery_2026-10-05.json')['result']
assert recovery['expected_pull_failure'] is True
assert recovery['pvc_identity_preserved'] is True
print(json.dumps({'evidence': 'historical serving and runtime adoption',
                  'paired_predictions': len(left), 'max_absolute_difference': differences,
                  'post_apply_plan': adoption['post_apply_plan']['result'],
                  'mlflow_image': adoption['post_apply_plan']['mlflow_image'],
                  'controlled_recovery_pvc_preserved': recovery['pvc_identity_preserved']}, indent=2))

{
  "evidence": "historical serving and runtime adoption",
  "paired_predictions": 3,
  "max_absolute_difference": {
    "predicted_fc": 0.0,
    "predicted_mw": 0.0
  },
  "post_apply_plan": "No changes",
  "mlflow_image": "ghcr.io/wanderson42/renewable-energy-mlops@sha256:233063b1cf82a2fb72426a2b09334ef31fb73b1db4bf7d1d5cfb8d4b4c0e657f",
  "controlled_recovery_pvc_preserved": true
}


## 8. Cliente isolado e o erro da UI Prefect

`make repro-client` inicia port-forwards e processos Prefect/Streamlit com Poetry, em loopback e portas próprias. Usa diretório, perfil e SQLite Prefect separados. O launcher permanece em primeiro plano e encerra os grupos de processos que criou ao receber Ctrl+C ou falhar; uma porta ocupada bloqueia o startup sem matar seu ocupante.

O primeiro gate confirmou os serviços, mas o operador encontrou erro na UI Prefect. A causa identificada foi `PREFECT_SERVER_API_AUTH_STRING=""`: na versão 3.8.6, a string vazia ativava o middleware de autenticação, embora os health checks passassem. Por isso, health HTTP sozinho não cobria a integração usada pela página.

A correção removeu os valores de autenticação herdados, manteve o perfil isolado e exigiu auth ausente/`None`. O gate passou a conferir a URL de API usada pela UI e uma consulta de dados sem header de autenticação. O novo recibo registrou HTTP 200 e `flow_runs_count=0`. O operador confirmou que o erro desapareceu e já havia informado funcionamento do dashboard.

O banco Prefect é novo: zero runs é esperado. Esta etapa não restaura a orquestração original nem demonstra flows/workers executados. Os recibos distinguem testes SDK/HTTP de relato manual no navegador; não há suíte automatizada de browser E2E neste gate.

Prefect sem Basic Auth é uma configuração local em loopback. As credenciais S3 continuam necessárias e privadas. A política de identidade, autenticação e exposição cloud será definida no blueprint, antes de qualquer eventual uso externo.

In [4]:
client = receipt('client_gate_recovery_2026-10-05.json')
assert client['lock_versions_match'] is True
assert client['model']['version'] == 17
assert client['model']['run_id'] == expected_run
assert client['prefect_settings_isolated'] is True
assert client['prefect_ui']['auth'] is None
assert client['prefect_ui']['data_query_http_status'] == 200
assert client['dashboard_health_http_status'] == 200
assert client['operator_browser_confirmation']['prefect_page_error_disappeared'] is True
print(json.dumps({'evidence': 'historical client recovery',
                  'validated_at': client['validated_at'], 'prefect_ui': client['prefect_ui'],
                  'dashboard_health_http_status': client['dashboard_health_http_status'],
                  'operator_browser_confirmation': client['operator_browser_confirmation']}, indent=2))

{
  "evidence": "historical client recovery",
  "validated_at": "2026-10-05T23:55:57.439609+00:00",
  "prefect_ui": {
    "api_url": "http://127.0.0.1:14200/api",
    "auth": null,
    "flow_runs_count": 0,
    "data_query_http_status": 200,
    "browser_e2e": "not exercised by this gate"
  },
  "dashboard_health_http_status": 200,
  "operator_browser_confirmation": {
    "reported_at": "2026-10-05T21:05:56-03:00",
    "prefect_page_error_disappeared": true,
    "scope": "manual operator report; no automated browser suite or flow execution evidenced"
  }
}


## 9. Evidências e limites do marco

| Resultado | Recibo público |
|---|---|
| Baseline da origem | [deployment_baseline](../../docs/evidence/deployment_baseline_2026-10-05.json) |
| Segundo cluster | [reproduction_cluster](../../docs/evidence/reproduction_cluster_2026-10-05.json) |
| Primeiro apply e plano | [terraform_bootstrap](../../docs/evidence/terraform_bootstrap_2026-10-05.json) |
| Snapshot e hashes | [snapshot_capture](../../docs/evidence/snapshot_capture_2026-10-05.json) |
| Restore e Registry | [snapshot_restore](../../docs/evidence/snapshot_restore_2026-10-05.json) |
| Inferência pareada | [serving_reproduction](../../docs/evidence/serving_reproduction_2026-10-05.json) |
| Falha de pull e recuperação | [deployment_recovery](../../docs/evidence/deployment_recovery_2026-10-05.json) |
| Build do runtime | [mlflow_runtime_build](../../docs/evidence/mlflow_runtime_build_2026-10-05.json) |
| Adoção e plano final | [mlflow_runtime_adoption](../../docs/evidence/mlflow_runtime_adoption_2026-10-05.json) |
| Cliente e correção Prefect | [client_gate_recovery](../../docs/evidence/client_gate_recovery_2026-10-05.json) |

Os recibos são históricos: um campo pendente no bootstrap ou restore descreve aquele momento, e não deve ser reinterpretado como estado final. Os testes têm escopos e revisões registrados nos recibos/runbook. Os 154 testes da consolidação da v0.3 são um marco anterior, não uma contagem atual de todos os testes desta branch.

O marco demonstra reprodução local com snapshot, artefatos e entradas definidos. Não demonstra reconstrução após perda total do host, retenção externa de backups, recuperação de PVC deletado, RTO/RPO, alta disponibilidade cloud ou generalização futura do modelo.

## 10. Próxima frente e critério de encerramento

O próximo trabalho é o blueprint Terraform AWS: rede, EKS, S3, RDS PostgreSQL e IAM, com fronteiras de acesso e configuração da aplicação documentadas. Na arquitetura proposta, S3 assume a função do object storage; RDS assume o backend PostgreSQL. O ambiente local permanece executável.

Sem conta autenticada, a validação prevista inclui formatação, validação de configuração, análise estática e testes com mocks. Um `terraform plan` real depende de acesso aos serviços AWS e não deve ser apresentado como executado quando houver apenas mocks. Não há recursos AWS criados neste marco.

Para encerrar a v1.0, ainda será necessário entregar e revisar o blueprint, alinhar README/runbooks/notebooks com o escopo final, obter os checks de integração da branch e consolidar a entrega por PR. O deployment local já fornece a trilha executável de evidências.

O acompanhamento longitudinal da v0.3 continua separado: revisões com 7 e 14 dias completos de geração observada e avaliação do mês fechado de outubro. Infraestrutura reproduzida não substitui essa avaliação nem aciona retreinamento automático.